In [1]:
import pandas as pd 
import re
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [2]:
df = pd.read_csv('/kaggle/input/datasets/saurabhshahane/fake-news-classification/WELFake_Dataset.csv')
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [3]:
df = df.drop('Unnamed: 0', axis=1)

In [4]:
df.shape

(72134, 3)

In [5]:
df.isnull().sum()

title    558
text      39
label      0
dtype: int64

In [6]:
df = df.dropna(subset=['text'])
df['title'] = df['title'].fillna("no_title")

In [7]:
df['content'] = df['title'] + " " + df['text']
df = df.drop(columns=['title', 'text'])
df.head()

,label,content
0,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1,1,no_title Did they post their votes for Hillary...
2,1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,1,SATAN 2: Russia unvelis an image of its terrif...


In [8]:
df = df.sample(frac=1)
df.head()

,label,content
33473,1,HE’S BAAACK! JUDGE NAPOLITANO On FOX News Not ...
51717,0,U.N. calls on Iran to resolve prisoner hunger ...
29584,1,US State Department Talking Head Transforms in...
48605,1,LOL! TRUMP Responds To RACE-OBSESSED Congressm...
52422,1,US Supreme Court justice groped female lawyer ...


In [9]:
df.reset_index(inplace=True)
df.head()

,index,label,content
0,33473,1,HE’S BAAACK! JUDGE NAPOLITANO On FOX News Not ...
1,51717,0,U.N. calls on Iran to resolve prisoner hunger ...
2,29584,1,US State Department Talking Head Transforms in...
3,48605,1,LOL! TRUMP Responds To RACE-OBSESSED Congressm...
4,52422,1,US Supreme Court justice groped female lawyer ...


In [10]:
df = df.drop('index', axis=1)

In [11]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    
    return " ".join(words)

In [12]:
df['content'] = df['content'].apply(clean_text)
df['content'] = df['content'].fillna("").astype(str)
df['content']

0        he’s baaack judge napolitano fox news backing ...
1        calls iran resolve prisoner hunger strike gene...
2        state department talking head transforms al qa...
3        lol trump responds raceobsessed congressman le...
4        supreme court justice groped female lawyer rep...
                               ...                        
72090    obama away free internet deems worthy “the int...
72091    china pakistan look including afghanistan bill...
72092    uber criminal investigation law enforcementeva...
72093    eric trump candidate investigation ‘unthinkabl...
72094    comment censored news black woman kills yo whi...
Name: content, Length: 72095, dtype: object

In [13]:
X = df['content']
y = df['label']

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), len(y_train))  
print(len(X_test), len(y_test)) 

57676 57676
14419 14419


In [15]:
X_train = X_train.fillna("").astype(str)
X_test = X_test.fillna("").astype(str)

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

In [17]:
Xv_train = vectorizer.fit_transform(X_train)

In [18]:
Xv_test = vectorizer.transform(X_test) 

In [19]:
print(Xv_train.shape, len(y_train))  
print(Xv_test.shape, len(y_test))

(57676, 5000) 57676
(14419, 5000) 14419


In [20]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(Xv_train, y_train)

LogisticRegression(max_iter=1000)

In [21]:
pred_lr = lr.predict(Xv_test)

In [22]:
from sklearn.metrics import classification_report
print(classification_report(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.95      0.94      0.94      6962
           1       0.94      0.95      0.95      7457

    accuracy                           0.95     14419
   macro avg       0.95      0.95      0.95     14419
weighted avg       0.95      0.95      0.95     14419



In [23]:
from sklearn.svm import LinearSVC

svm = LinearSVC(C=1.0)
svm.fit(Xv_train, y_train)

LinearSVC()

In [24]:
pred_svm = svm.predict(Xv_test)
print(classification_report(y_test, pred_svm))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      6962
           1       0.95      0.96      0.95      7457

    accuracy                           0.95     14419
   macro avg       0.95      0.95      0.95     14419
weighted avg       0.95      0.95      0.95     14419

